Data Ingestion

document datastructure

In [1]:
from langchain_core.documents import Document 

In [2]:
doc=Document(
    page_content="this is the main text content iam using to create RAG",
    metadata={
        "source":"example.txt",
        "pages":1,
        "author":"Anupam DS",
        "date_created":"2025-10-25"
    }
)
doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Anupam DS', 'date_created': '2025-10-25'}, page_content='this is the main text content iam using to create RAG')

In [3]:
## Create a simple txt file 
import os 
os.makedirs("../data/text_files",exist_ok=True)

In [4]:
sample_texts={
    "../data/text_files/python_intro.txt":"""Python Programing Introduction
    Python is high-level, interpreted programing language known for its simplicity and readability.
    Created by Guido van Rossum and first released in 1991 , Python has become one of the most popular programming languages in the world .
    
    Key Features :
    - Easy to learn and use 
    - Extensive standard library 
    - Cross -platform compatibility
    - Strong community support 
    
    Python is widely used in web development , data science , artificial intelligence , and automation.""",
    
    "../data/text_files/machine_learning.txt": """Machine Learning Basics
    Machine learning is a subset of artficial intelligence that enables systems to learn and improve from experience
    without being explicitly programmed . It focuses on developing computer programs that 
    can acces data and use it to learn for thenselves.
    
    Tyoes of Machine Learning:
    1.Supervised Learning : Learning with labeled data 
    2. Unsupervised Learning : Finding patterns in unlabeled data
    3. Reinforcement Learning : Learning through rewards and penalties 

    Applications include image recofnition, speech processing and recommendation systems
    """
}

for filepath, content in sample_texts.items():
    with open (filepath, 'w' , encoding="utf-8") as f:
        f.write(content)

print(" Sample text files created. ")

 Sample text files created. 


In [5]:
##TextLoader 

from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import TextLoader

loader=TextLoader("../data/text_files/python_intro.txt",encoding="utf-8")
document=loader.load()
print(document)


[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programing Introduction\n    Python is high-level, interpreted programing language known for its simplicity and readability.\n    Created by Guido van Rossum and first released in 1991 , Python has become one of the most popular programming languages in the world .\n\n    Key Features :\n    - Easy to learn and use \n    - Extensive standard library \n    - Cross -platform compatibility\n    - Strong community support \n\n    Python is widely used in web development , data science , artificial intelligence , and automation.')]


In [6]:
## Directory Loader
from langchain_community.document_loaders import DirectoryLoader

#load all the text files from the directory 
dir_loader=DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt", ##Pattern to match files
    loader_cls=TextLoader, ##loader class to use
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=False
)

documents=dir_loader.load()
documents

[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n    Machine learning is a subset of artficial intelligence that enables systems to learn and improve from experience\n    without being explicitly programmed . It focuses on developing computer programs that \n    can acces data and use it to learn for thenselves.\n\n    Tyoes of Machine Learning:\n    1.Supervised Learning : Learning with labeled data \n    2. Unsupervised Learning : Finding patterns in unlabeled data\n    3. Reinforcement Learning : Learning through rewards and penalties \n\n    Applications include image recofnition, speech processing and recommendation systems\n    '),
 Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='Python Programing Introduction\n    Python is high-level, interpreted programing language known for its simplicity and readability.\n    Created by Guido van Rossum and first released in 1991 , Python

In [7]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

##load all the text files from the directory

dir_loader=DirectoryLoader(
    "../data/pdf",
    glob='**/*.pdf', ##Pattern to match files
    loader_cls=PyMuPDFLoader, ##loader class to use
    show_progress=False

)
pdf_document=dir_loader.load()
pdf_document

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'source': '..\\data\\pdf\\1706.03762v7.pdf', 'file_path': '..\\data\\pdf\\1706.03762v7.pdf', 'total_pages': 15, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'trapped': '', 'modDate': 'D:20240410211143Z', 'creationDate': 'D:20240410211143Z', 'page': 0}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\n

In [17]:
# %pip install --upgrade --quiet langchain-community unstructured openpyxl

In [18]:
# !pip install unstructured openpyxl

In [ ]:
# pip install "unstructured[xlsx]"

Note: you may need to restart the kernel to use updated packages.


d:\Documents\PROJECTS\RAG\.venv\Scripts\python.exe: No module named pip


In [ ]:
# pip install python-pptx xlrd pillow

Note: you may need to restart the kernel to use updated packages.


d:\Documents\PROJECTS\RAG\.venv\Scripts\python.exe: No module named pip


In [16]:
# from langchain_community.document_loaders import UnstructuredExcelLoader

# loader = UnstructuredExcelLoader("data\excel\Financial Sample.xlsx", mode="elements")
# docs = loader.load()

# print(len(docs))

# docs

Embedding And VectorStoreDB

In [19]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid 
from typing import List,Dict,Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [22]:
class Embeddingmanager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self,model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        Args:
        model_name: HuggingFace model name for sentence embeddings
        """

        self.model_name = model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}:{e}")
            raise
    
    def generate_embedding(self,texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
        texts: List of text strings to embed

        Returns :
            numpy array of embeddings with shape (len(texts), embedding_dim)

        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings=self.model.encode(texts,show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings:shape}")
        return embeddings 
    


##initialize the embedding manager

embedding_manager=Embeddingmanager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


d:\Documents\PROJECTS\RAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ACER\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to re

Model loaded successfully. Embedding dimension: 384


Vector Store

In [24]:
class VectorStore:
    """Manage document embeddings in a ChromDB vector store"""
    
    def __init__(self, collection_name:str = "pdf_documents",presist_directory: str="../data/vector_store"):
        """
        Initialize the vectore Store

        Args:
            collection_name : Name of the ChromaDB collection
            presist_directory: Directory to presist the vector store
    
        """
        self.collection_name = collection_name
        self.presist_directory = presist_directory
        self.client=None
        self.collection = None
        self._initialize_store()
    
    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""

        try:
            #Create presistent ChromaDB client
            os.makedirs(self.presist_directory,exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.presist_directory)

            #Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized.collection: {self.collection_name}")
            print(f"Existing documents in collection : { self.collection.count()}")
        
        except Exception as e:
            print(f"Error initializing vector store : {e}")
            raise

    def add_documents(self, documents: List[Any],embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store 
        
        Args:
            documents : List of Langchain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")

        #Prepare data for chromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc,embedding) in enumerate(zip(documents,embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #prepare metadata
            metadata= dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            #Document Content 
            documents_text.append(embedding.tolist)

            #Embedding 
            embeddings_list.append(embedding.tolist())
        
        #Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Succesfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store : {e}")
            raise

VectorStore=VectorStore()
VectorStore


        

Vector store initialized.collection: pdf_documents
Existing documents in collection : 0


In [25]:
chunks

NameError: name 'chunks' is not defined